In [2]:
# ==============================================
# 1. 导入所有依赖
# ==============================================
import pandas as pd
import numpy as np
import re
from openpyxl.styles import PatternFill, Alignment, Border, Side, Font
from openpyxl.utils import get_column_letter

# ==============================================
# 2. 【全局配置区】所有参数统一在这里修改
# ==============================================
# 基础参数
台数 = 4
selected_model = "S1"
selected_mode = 2  # 充放模式

# 电价参数
放电含税电价 = 1.0405
充电含税电价 = 0.3128

# 效率参数
系统效率_收益测算 = 0.885  # 用于收益测算
系统效率_回收期 = 0.865    # 用于回收期测算

# 费率参数
运维成本率 = 0.01
所得税率 = 0.05
年利率 = 0.04

# 电气参数
安全系数 = 0.85
变压器容量_kVA = 12550

# 衰减参数
DOD = 1.0
年衰减率 = 0.025

# 商务参数
商务模式 = "EMC"
折扣 = 0.8
居间费率 = 0.13 # 全局居间费率
上午平段限制放电量比例 = 1.0
下午限制放电量比例 = 1.0
下午平段充电倍率 = 0
尖峰前高峰放电倍率 = 1.0

# ==============================================
# 3. 设备规格表
# ==============================================
device_specs = pd.DataFrame([
    ["S1", 1, 50, 225, 235, 11.6, 3.5, "0.25C"],
    ["S1", 2, 100, 225, 235, 11.6, 4, "0.5C"],
    ["S2", 1, 65, 261, 271, 14, 3.5, "0.25C"],
    ["S2", 2, 130, 261, 271, 14, 3.915, "0.5C"],
    ["X3", 1, 215, 836, 876, 44.308, "", "0.25C"],
    ["X3", 2, 418, 836, 876, 44.308, "", "0.5C"],
], columns=["设备型号", "充放模式", "额定功率(kW)", "额定容量(kWh)", "实际容量(kWh)", "设备成本(万/台)", "预计施工费(万/台)", "备注"])

# ==============================================
# 4. 全局样式定义
# ==============================================
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
thick_border = Border(
    left=Side(style='medium'), right=Side(style='medium'),
    top=Side(style='medium'), bottom=Side(style='medium')
)
center_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
title_font = Font(name='微软雅黑', size=14, bold=True)
header_font = Font(name='微软雅黑', size=10, bold=True)
data_font = Font(name='微软雅黑', size=9)
fill_header_yellow = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
fill_charge = PatternFill(start_color='CCFFCC', end_color='CCFFCC', fill_type='solid')
fill_flat = PatternFill(start_color='CCE5FF', end_color='CCE5FF', fill_type='solid')
fill_peak = PatternFill(start_color='FFE5CC', end_color='FFE5CC', fill_type='solid')
light_yellow_fill = PatternFill(start_color='FFFF99', end_color='FFFF99', fill_type='solid')

# ==============================================
# 5. 派生参数计算
# ================================ww============
spec_row = device_specs[(device_specs["设备型号"] == selected_model) & (device_specs["充放模式"] == selected_mode)].iloc[0]
设备规格_功率_kW = spec_row["额定功率(kW)"]
设备规格_标定容量_kWh = spec_row["额定容量(kWh)"]
设备规格_实际容量_kWh = spec_row["实际容量(kWh)"]
设备成本_万_台 = spec_row["设备成本(万/台)"]
施工费_万_台 = spec_row["预计施工费(万/台)"] if pd.notna(spec_row["预计施工费(万/台)"]) else 0.0

系统功率_kW = 设备规格_功率_kW * 台数
系统额定电量_kWh = 设备规格_标定容量_kWh * 台数
系统实际电量_kWh = 设备规格_实际容量_kWh * 台数

放电不含税电价 = 放电含税电价 / 1.13
充电不含税电价 = 充电含税电价 / 1.13
功率上限_kW = 安全系数 * 变压器容量_kVA
临界容量 = 变压器容量_kVA * 0.625
日衰减率 = 年衰减率 / 365
充放模式 = selected_mode
电价折扣 = 折扣

# 新增全局变量 initial_invest_excl 和 cost_per_kwh
cost_per_kwh = 1.05
initial_invest_excl = 系统额定电量_kWh * cost_per_kwh / 10 / 1.13


# ==============================================
# 6. 时段定义
# ==============================================
上午固定平段列表 = ["09:00:00", "09:15:00", "09:30:00", "09:45:00", "10:00:00", "10:15:00", "10:30:00", "10:45:00"]
历史规则1_时段列表 = ["15:00:00", "15:15:00", "15:30:00", "15:45:00"]
历史规则2_时段列表 = ["16:00:00", "16:15:00", "16:30:00", "16:45:00", "17:00:00", "17:15:00", "17:30:00", "17:45:00", "18:00:00", "18:15:00", "18:30:00", "18:45:00", "19:00:00", "19:15:00", "19:30:00", "19:45:00", "20:00:00", "20:15:00", "20:30:00", "20:45:00", "21:00:00", "21:15:00", "21:30:00", "21:45:00", "22:00:00", "22:15:00", "22:30:00", "22:45:00"]
历史规则3_时段列表 = ["16:00:00", "16:15:00", "16:30:00", "16:45:00", "17:00:00", "17:15:00", "17:30:00", "17:45:00", "18:00:00", "18:15:00", "18:30:00", "18:45:00", "19:00:00", "19:15:00", "19:30:00", "19:45:00"]
历史规则4_时段列表 = ["15:00:00", "15:15:00", "15:30:00", "15:45:00", "16:00:00", "16:15:00", "16:30:00", "16:45:00", "17:00:00", "17:15:00", "17:30:00", "17:45:00", "18:00:00", "18:15:00", "18:30:00", "18:45:00"]
新规则1_时段列表 = ["15:00:00", "15:15:00", "15:30:00", "15:45:00"]
新规则2_时段列表 = ["16:00:00", "16:15:00", "16:30:00", "16:45:00", "17:00:00", "17:15:00", "17:30:00", "17:45:00", "18:00:00", "18:15:00", "18:30:00", "18:45:00", "19:00:00", "19:15:00", "19:30:00", "19:45:00"]
新规则3_时段列表 = ["20:00:00", "20:15:00", "20:30:00", "20:45:00", "21:00:00", "21:15:00", "21:30:00", "21:45:00", "22:00:00", "22:15:00", "22:30:00", "22:45:00"]
新规则23点时段列表 = ["23:00:00", "23:15:00", "23:30:00", "23:45:00"]
需求_12月下旬_19点时段 = ["19:00:00", "19:15:00", "19:30:00", "19:45:00"]
上午尖峰时段列表 = 上午固定平段列表.copy()
下午尖峰时段列表 = ["15:00:00", "15:15:00", "15:30:00", "15:45:00", "16:00:00", "16:15:00", "16:30:00", "16:45:00"]

# ==============================================
# 7. 辅助函数
# ==============================================
def calculate_payback_for_combination(discount, jujian):
    discharge_price_shuiqian = 放电不含税电价
    charge_price_shuiqian = 充电不含税电价
    discharge_emic = 放电含税电价
    charge_emic = 充电含税电价
    run_days = 总计_折算天数 # 从月度汇总中获取
    edingdianliang = 系统额定电量_kWh
    chunengdianliang = 系统实际电量_kWh
    taishu = 台数
    low_pressure_default_cost = 设备成本_万_台
    construction_default_cost = 施工费_万_台

    suodeshui = 所得税率
    dianjiazhekou = discount
    nianlixi = 年利率
    baoxianbili = 0.0015
    fangdianshendu = 1
    xitongxiaolv = 系统效率_回收期
    xiaoxiangshuilv = 0.13
    JUJIAN = jujian # 使用传入的居间费率
    unit_om_cost = 运维成本率

    di9diya = low_pressure_default_cost * taishu
    di9shigong = construction_default_cost * taishu
    di1diya = di9diya / edingdianliang * 10
    di1shigong = di9shigong / edingdianliang * 10

    invest_yr9 = round(0.4 * edingdianliang / 10 / 1.13, 4)
    invest_tax_yr1 = (di1diya * edingdianliang / 10) + (di1shigong * edingdianliang / 10) + (JUJIAN * edingdianliang / 10)
    invest_no_tax_yr1 = (di1diya * edingdianliang / 10 / 1.13) + (di1shigong * edingdianliang / 10) + (JUJIAN * edingdianliang / 10)
    input_tax = round(invest_tax_yr1 - invest_no_tax_yr1, 4)

    jujian_yr9 = [0.0] * 17
    jujian_yr9[0] = round(edingdianliang * JUJIAN / 10, 4)

    power_station_invest = [0.0] * 17
    power_station_invest[0] = round(edingdianliang * di1diya / 10, 4)
    power_station_invest[8] = round(invest_yr9 * 1.13, 4)

    construction_invest = [0.0] * 17
    construction_invest[0] = di9shigong

    battery_ratio = [0.0,0.965,0.911,0.877,0.848,0.823,0.800,0.778,0.758,0.965,0.911,0.877,0.848,0.823,0.800,0.778,0.758]

    charge_kwh = [0.0] * 17
    for n in range(1, 17):
        charge_kwh[n] = round((run_days * chunengdianliang * fangdianshendu)*2 / 10000 / (1 - (1 - xitongxiaolv)/2) * battery_ratio[n], 2)

    discharge_kwh = [0.0] * 17
    for n in range(1, 17):
        discharge_kwh[n] = round(xitongxiaolv * charge_kwh[n], 2)

    cash_in = [0.0] * 17
    for n in range(1, 17):
        cash_in[n] = round((discharge_kwh[n] * discharge_emic - charge_kwh[n] * charge_emic) * dianjiazhekou, 2)

    electricity_income = cash_in.copy()
    operation_cost = [0.0] * 17
    for n in range(1, 17):
        operation_cost[n] = round(unit_om_cost * 20 * taishu, 2)

    insurance_cost = [0.0] * 17
    for n in range(1, 17):
        insurance_cost[n] = round(invest_no_tax_yr1 * baoxianbili, 4)

    output_tax = [0.0] * 17
    for n in range(1, 17):
        output_tax[n] = round(cash_in[n] - cash_in[n]/(1 + xiaoxiangshuilv), 4)

    vat_tax = [0.0] * 17
    vat_tax[0] = round(output_tax[0] - input_tax, 4)
    for n in range(1, 17):
        vat_tax[n] = round(output_tax[n], 4)

    pay_vat = [0.0] * 17
    for n in range(1, 17):
        sum_vat = sum(vat_tax[:n+1])
        sum_pay_before = sum(pay_vat[:n])
        if sum_vat > 0 and sum_pay_before == 0:
            pay_vat[n] = round(sum_vat, 4)
        elif sum_vat > 0 and sum_pay_before > 0:
            pay_vat[n] = round(vat_tax[n], 4)
        else:
            pay_vat[n] = 0.0

    tax_surcharge = [0.0] * 17
    for n in range(1, 17):
        tax_surcharge[n] = round(cash_in[n] * 3/10000 + pay_vat[n] * 0.12, 4)

    depreciation = [0.0] * 17
    depr_yr1_8 = round(invest_no_tax_yr1 * 0.95 / 8, 4)
    depr_yr9_16 = round(power_station_invest[8] * 0.95 / (1.13 * 8), 4)
    for n in range(1, 17):
        if 1 <= n <= 8:
            depreciation[n] = depr_yr1_8
        elif 9 <= n <= 16:
            depreciation[n] = depr_yr9_16

    excel_cash_out = [0.0] * 17
    excel_net_cash_flow = [0.0] * 17
    excel_cum_cash_flow = [0.0] * 17
    excel_interest = [0.0] * 17
    excel_income_tax = [0.0] * 17

    excel_cash_out[0] = round(di9diya + di9shigong + jujian_yr9[0], 4)
    excel_net_cash_flow[0] = round(-excel_cash_out[0], 4)
    excel_cum_cash_flow[0] = excel_net_cash_flow[0]

    for n in range(1, 17):
        if excel_cum_cash_flow[n-1] < 0:
            excel_interest[n] = round(abs(excel_cum_cash_flow[n-1]) * nianlixi, 4)
        else:
            excel_interest[n] = 0.0

        taxable_base = (
            electricity_income[n] - operation_cost[n] - insurance_cost[n]
            - tax_surcharge[n] - excel_interest[n] - depreciation[n] - output_tax[n]
        )
        excel_income_tax[n] = round(taxable_base * suodeshui, 4)

        excel_cash_out[n] = round(
            power_station_invest[n] + jujian_yr9[n] + operation_cost[n] + insurance_cost[n]
            + tax_surcharge[n] + excel_interest[n] + excel_income_tax[n] + pay_vat[n],
            4
        )

        excel_net_cash_flow[n] = round(cash_in[n] - excel_cash_out[n], 4)
        excel_cum_cash_flow[n] = round(excel_cum_cash_flow[n-1] + excel_net_cash_flow[n], 4)

    break_even_year = None
    for year_idx in range(17):
        if excel_cum_cash_flow[year_idx] >= 0:
            break_even_year = year_idx
            break

    if break_even_year is not None and break_even_year > 0:
        m = break_even_year - 1
        numerator = abs(excel_cum_cash_flow[m])
        denominator = excel_cum_cash_flow[m+1] - excel_cum_cash_flow[m]
        if denominator != 0:
            pb_value = numerator / denominator
            actual_value = m + pb_value
            return round(actual_value, 2)
    return None

def generate_payback_table():
    discharge_price_shuiqian = 放电不含税电价
    charge_price_shuiqian = 充电不含税电价
    discharge_emic = 放电含税电价
    charge_emic = 充电含税电价
    run_days = 总计_折算天数 # 从月度汇总中获取
    edingdianliang = 系统额定电量_kWh
    chunengdianliang = 系统实际电量_kWh
    taishu = 台数
    low_pressure_default_cost = 设备成本_万_台
    construction_default_cost = 施工费_万_台

    suodeshui = 所得税率
    dianjiazhekou = 电价折扣
    nianlixi = 年利率
    baoxianbili = 0.0015
    fangdianshendu = 1
    xitongxiaolv = 系统效率_回收期
    xiaoxiangshuilv = 0.13
    JUJIAN = 居间费率 # 使用全局变量居间费率
    unit_om_cost =运维成本率

    di9diya = low_pressure_default_cost * taishu
    di9shigong = construction_default_cost * taishu
    di1diya = di9diya / edingdianliang * 10
    di1shigong = di9shigong / edingdianliang * 10

    invest_yr9 = round(0.4 * edingdianliang / 10 / 1.13, 4)
    invest_tax_yr1 = (di1diya * edingdianliang / 10) + (di1shigong * edingdianliang / 10) + (JUJIAN * edingdianliang / 10)
    invest_no_tax_yr1 = (di1diya * edingdianliang / 10 / 1.13) + (di1shigong * edingdianliang / 10) + (JUJIAN * edingdianliang / 10)
    input_tax = round(invest_tax_yr1 - invest_no_tax_yr1, 4)

    jujian_yr9 = [0.0] * 17
    jujian_yr9[0] = round(edingdianliang * JUJIAN / 10, 4)

    power_station_invest = [0.0] * 17
    power_station_invest[0] = round(edingdianliang * di1diya / 10, 4)
    power_station_invest[8] = round(invest_yr9 * 1.13, 4)

    construction_invest = [0.0] * 17
    construction_invest[0] = di9shigong

    battery_ratio = [0.0,0.965,0.911,0.877,0.848,0.823,0.800,0.778,0.758,0.965,0.911,0.877,0.848,0.823,0.800,0.778,0.758]
    battery_percent = [f"{v*100:.1f}%" if v !=0 else "0.0%" for v in battery_ratio]

    charge_kwh = [0.0] * 17
    for n in range(1, 17):
        charge_kwh[n] = round((run_days * chunengdianliang * fangdianshendu)*2 / 10000 / (1 - (1 - xitongxiaolv)/2) * battery_ratio[n], 2)

    discharge_kwh = [0.0] * 17
    for n in range(1, 17):
        discharge_kwh[n] = round(xitongxiaolv * charge_kwh[n], 2)

    cash_in = [0.0] * 17
    for n in range(1, 17):
        cash_in[n] = round((discharge_kwh[n] * discharge_emic - charge_kwh[n] * charge_emic) * dianjiazhekou, 2)

    electricity_income = cash_in.copy()
    operation_cost = [0.0] * 17
    for n in range(1, 17):
        operation_cost[n] = round(unit_om_cost * 20 * taishu, 2)

    insurance_cost = [0.0] * 17
    for n in range(1, 17):
        insurance_cost[n] = round(invest_no_tax_yr1 * baoxianbili, 4)

    output_tax = [0.0] * 17
    for n in range(1, 17):
        output_tax[n] = round(cash_in[n] - cash_in[n]/(1 + xiaoxiangshuilv), 4)

    vat_tax = [0.0] * 17
    vat_tax[0] = round(output_tax[0] - input_tax, 4)
    for n in range(1, 17):
        vat_tax[n] = round(output_tax[n], 4)

    pay_vat = [0.0] * 17
    for n in range(1, 17):
        sum_vat = sum(vat_tax[:n+1])
        sum_pay_before = sum(pay_vat[:n])
        if sum_vat > 0 and sum_pay_before == 0:
            pay_vat[n] = round(sum_vat, 4)
        elif sum_vat > 0 and sum_pay_before > 0:
            pay_vat[n] = round(vat_tax[n], 4)
        else:
            pay_vat[n] = 0.0

    tax_surcharge = [0.0] * 17
    for n in range(1, 17):
        tax_surcharge[n] = round(cash_in[n] * 3/10000 + pay_vat[n] * 0.12, 4)

    depreciation = [0.0] * 17
    depr_yr1_8 = round(invest_no_tax_yr1 * 0.95 / 8, 4)
    depr_yr9_16 = round(power_station_invest[8] * 0.95 / (1.13 * 8), 4)
    for n in range(1, 17):
        if 1 <= n <= 8:
            depreciation[n] = depr_yr1_8
        elif 9 <= n <= 16:
            depreciation[n] = depr_yr9_16

    excel_cash_out = [0.0] * 17
    excel_net_cash_flow = [0.0] * 17
    excel_cum_cash_flow = [0.0] * 17
    excel_interest = [0.0] * 17
    excel_income_tax = [0.0] * 17

    excel_cash_out[0] = round(di9diya + di9shigong + jujian_yr9[0], 4)
    excel_net_cash_flow[0] = round(-excel_cash_out[0], 4)
    excel_cum_cash_flow[0] = excel_net_cash_flow[0]

    for n in range(1, 17):
        if excel_cum_cash_flow[n-1] < 0:
            excel_interest[n] = round(abs(excel_cum_cash_flow[n-1]) * nianlixi, 4)
        else:
            excel_interest[n] = 0.0

        taxable_base = (
            electricity_income[n] - operation_cost[n] - insurance_cost[n]
            - tax_surcharge[n] - excel_interest[n] - depreciation[n] - output_tax[n]
        )
        excel_income_tax[n] = round(taxable_base * suodeshui, 4)

        excel_cash_out[n] = round(
            power_station_invest[n] + jujian_yr9[n] + operation_cost[n] + insurance_cost[n]
            + tax_surcharge[n] + excel_interest[n] + excel_income_tax[n] + pay_vat[n],
            4
        )

        excel_net_cash_flow[n] = round(cash_in[n] - excel_cash_out[n], 4)
        excel_cum_cash_flow[n] = round(excel_cum_cash_flow[n-1] + excel_net_cash_flow[n], 4)

    payback_yearly = [0.0] * 17
    final_payback_text = "全周期无法回收投资" # 用于DataFrame中的文字描述

    break_even_year = None
    for year_idx in range(17):
        if excel_cum_cash_flow[year_idx] >= 0:
            break_even_year = year_idx
            break

    actual_final = 0.0 # 用于返回的数值
    if break_even_year is not None:
        if break_even_year == 0:
            actual_final = 0.0
            final_payback_text = "0.00年（已回收）"
            for year_idx in range(17):
                payback_yearly[year_idx] = "0.00年（已回收）"
        else:
            m = break_even_year - 1
            numerator = abs(excel_cum_cash_flow[m])
            denominator = excel_cum_cash_flow[m+1] - excel_cum_cash_flow[m]
            if denominator != 0:
                calculated_pb_value = numerator / denominator
                actual_final = round(m + calculated_pb_value, 2) # 数值型回收期
                final_payback_text = f"{actual_final:.2f}年（已回收）"
            else:
                calculated_pb_value = 0.0 # 或者其他逻辑
                actual_final = round(m + calculated_pb_value, 2)
                final_payback_text = f"{actual_final:.2f}年（已回收）"


            for year_idx in range(17):
                if year_idx < break_even_year:
                    if year_idx < 16:
                        # 动态计算预测回收期，但这里只用于显示，实际回收期是actual_final
                        curr_cum = excel_cum_cash_flow[year_idx]
                        next_cum = excel_cum_cash_flow[year_idx+1]
                        if next_cum - curr_cum != 0:
                            dynamic_pb = abs(curr_cum) / (next_cum - curr_cum)
                            payback_yearly[year_idx] = f"{dynamic_pb:.2f}年（预测）"
                        else:
                            payback_yearly[year_idx] = "未回收"
                    else:
                        payback_yearly[year_idx] = "未回收"
                else:
                    payback_yearly[year_idx] = final_payback_text # 所有回收期后的年份都显示最终回收期
    else:
        for year_idx in range(17):
            payback_yearly[year_idx] = "未回收"


    recovery_actual_str_list = []
    for year_idx in range(17):
        if excel_cum_cash_flow[year_idx] >= 0 and year_idx > 0:
            m = year_idx - 1
            if excel_cum_cash_flow[year_idx] - excel_cum_cash_flow[m] != 0:
                pb_val_interp = m + abs(excel_cum_cash_flow[m]) / (excel_cum_cash_flow[year_idx] - excel_cum_cash_flow[m])
                recovery_actual_str_list.append(f"{pb_val_interp:.2f}")
            else:
                recovery_actual_str_list.append("无法计算")
        elif year_idx == 0 and excel_cum_cash_flow[year_idx] >= 0:
            recovery_actual_str_list.append("0.00")
        else:
            recovery_actual_str_list.append(" ") # 回收期前的显示为空

    # 确保 recovery_actual_str_list 有 17 个元素
    while len(recovery_actual_str_list) < 17:
        recovery_actual_str_list.append(" ")


    year_headers = ['项目'] + [f'第{i}年' for i in range(0,17)] + ['合计']
    columns = year_headers

    data = [
        ['电池容量', *battery_percent, '-'],
        ['充电量（万度）', *charge_kwh, round(sum(charge_kwh), 4)],
        ['放电量（万度）', *discharge_kwh, round(sum(discharge_kwh), 4)],
        ['现金流入（电费收入）', *cash_in, round(sum(cash_in), 4)],
        ['电费收入', *electricity_income, round(sum(electricity_income), 4)],
        ['现金流出', *excel_cash_out, round(sum(excel_cash_out), 4)],
        ['电站投资', *power_station_invest, round(sum(power_station_invest), 4)],
        ['施工', *construction_invest, round(sum(construction_invest), 4)],
        ['第9年居间', *jujian_yr9, round(sum(jujian_yr9), 4)],
        ['运营成本', *operation_cost, round(sum(operation_cost), 4)],
        ['保险费用', *insurance_cost, round(sum(insurance_cost), 4)],
        ['税金及附加', *tax_surcharge, round(sum(tax_surcharge), 4)],
        ['利息', *excel_interest, round(sum(excel_interest), 4)],
        ['税费（所得税，非绝对值）', *excel_income_tax, round(sum(excel_income_tax), 4)],
        ['支付增值税', *pay_vat, round(sum(pay_vat), 4)],
        ['现金净流入', *excel_net_cash_flow, round(sum(excel_net_cash_flow), 4)],
        ['累计现金流量', *excel_cum_cash_flow, round(excel_cum_cash_flow[-1], 4)],
        ['项目折旧', *depreciation, round(sum(depreciation), 4)],
        ['增值税', *vat_tax, round(sum(vat_tax), 4)],
        ['销项税', *output_tax, round(sum(output_tax), 4)],
        ['静态投资回收期', *payback_yearly, final_payback_text],
        ['回收周期实测', *recovery_actual_str_list, f"{actual_final:.2f}" if actual_final is not None else "无法回收"],
    ]
    return pd.DataFrame(data, columns=columns), actual_final # 返回DataFrame和实际回收期数值

# ==============================================
# 8. 主计算流程
# ==============================================
# 读取原始数据
df_raw = pd.read_excel("原始数据.xlsx", sheet_name="负荷-原始数据", usecols="B:D", header=0)
df_raw.columns = ["日期", "时间", "瞬时有功"]
df_raw = df_raw.dropna(subset=["日期", "时间", "瞬时有功"])
df_raw["日期"] = pd.to_datetime(df_raw["日期"]).dt.date
df_raw["时间"] = pd.to_datetime(df_raw["时间"]).dt.strftime("%H:%M:%S")

# 生成宽表
df_pivot = df_raw.pivot(index="日期", columns="时间", values="瞬时有功").reset_index()
all_time_cols = sorted([col for col in df_pivot.columns if col != "日期"])
df_pivot = df_pivot[["日期"] + all_time_cols]
df_pivot.insert(loc=0, column="序号", value=1)

# 计算基础列
df_pivot["最大需量（日）"] = df_pivot[all_time_cols].max(axis=1)
df_pivot["充电保护值"] = 功率上限_kW

valid_morning_cols = [col for col in 上午尖峰时段列表 if col in all_time_cols]
valid_afternoon_cols = [col for col in 下午尖峰时段列表 if col in all_time_cols]
df_pivot["上午尖峰最小值"] = df_pivot[valid_morning_cols].min(axis=1)
df_pivot["下午尖峰最小值"] = df_pivot[valid_afternoon_cols].min(axis=1)
df_pivot["尖最小功率"] = df_pivot[["上午尖峰最小值", "下午尖峰最小值"]].min(axis=1)
df_pivot = df_pivot.drop(columns=["上午尖峰最小值", "下午尖峰最小值"])

df_pivot["1充放电容量（上午）"] = 系统实际电量_kWh * 上午平段限制放电量比例
df_pivot["1充放电容量（下午）"] = 系统实际电量_kWh * 下午限制放电量比例

初始容量 = 系统实际电量_kWh
日衰减量 = 初始容量 * 日衰减率
df_pivot["充电容量"] = 初始容量 - 日衰减量 * pd.Series(range(len(df_pivot)))
df_pivot["系统留存"] = 0.00

month_series = df_pivot['日期'].apply(lambda x: x.month)
md_series = df_pivot['日期'].apply(lambda x: x.strftime("%m-%d"))

# 核心充放电逻辑
放电保护 = 系统功率_kW * 0.1
充电值_df = pd.DataFrame(index=df_pivot.index, columns=all_time_cols)
放电值_df = pd.DataFrame(index=df_pivot.index, columns=all_time_cols)
for time_col in all_time_cols:
    充电值_df[time_col] = (df_pivot['充电保护值'] - df_pivot[time_col]).clip(lower=0).clip(upper=系统功率_kW)
    放电值_df[time_col] = (df_pivot[time_col] - 放电保护).clip(lower=0).clip(upper=系统功率_kW)

new_df = pd.DataFrame(index=df_pivot.index, columns=df_pivot.columns)
new_df['日期'] = df_pivot['日期']
new_df['序号'] = ''
formula_type_map = pd.DataFrame(index=df_pivot.index, columns=all_time_cols)

for row_idx in range(len(df_pivot)):
    current_month = month_series.iloc[row_idx]
    current_md = md_series.iloc[row_idx]
    is_7_to_9_month = 7 <= current_month <= 9
    is_2_to_11_month = 2 <= current_month <= 11

    for col_idx, time_col in enumerate(all_time_cols):
        cell_value = np.nan
        formula_type = None

        if time_col in 需求_12月下旬_19点时段 and current_md >= "12-16":
            cell_value = 放电值_df[time_col].iloc[row_idx]
            formula_type = 'flat_discharge'
        elif time_col in 新规则23点时段列表:
            if "01-01" <= current_md <= "06-30" or "10-01" <= current_md <= "12-31":
                cell_value = 充电值_df[time_col].iloc[row_idx]
                formula_type = 'charge'
            elif "07-01" <= current_md <= "09-30":
                cell_value = 放电值_df[time_col].iloc[row_idx]
                formula_type = 'flat_discharge'
        elif time_col in 新规则1_时段列表 and "02-01" <= current_md <= "12-15":
            cell_value = 充电值_df[time_col].iloc[row_idx]
            formula_type = 'charge'
        elif time_col in 新规则2_时段列表 and "09-01" <= current_md <= "12-15":
            cell_value = 放电值_df[time_col].iloc[row_idx]
            formula_type = 'flat_discharge'
        elif time_col in 新规则3_时段列表 and current_md >= "07-15":
            cell_value = 放电值_df[time_col].iloc[row_idx]
            formula_type = 'flat_discharge'
        elif time_col in 历史规则4_时段列表 and current_md >= "12-16":
            cell_value = 放电值_df[time_col].iloc[row_idx]
            formula_type = 'peak_discharge'
        elif time_col in 历史规则3_时段列表 and "07-15" <= current_md <= "08-31":
            cell_value = 放电值_df[time_col].iloc[row_idx]
            formula_type = 'peak_discharge'
        elif time_col in 历史规则2_时段列表 and current_md <= "07-14":
            cell_value = 放电值_df[time_col].iloc[row_idx]
            formula_type = 'flat_discharge'
        elif time_col in 历史规则1_时段列表 and current_md <= "01-31":
            cell_value = 放电值_df[time_col].iloc[row_idx]
            formula_type = 'peak_discharge'
        elif time_col in 上午固定平段列表:
            cell_value = 放电值_df[time_col].iloc[row_idx]
            formula_type = 'flat_discharge'
        else:
            if col_idx < 24:
                cell_value = 充电值_df[time_col].iloc[row_idx]
                formula_type = 'charge'
            elif 24 <= col_idx <= 35:
                if is_7_to_9_month:
                    cell_value = 充电值_df[time_col].iloc[row_idx]
                    formula_type = 'charge'
                else:
                    cell_value = 放电值_df[time_col].iloc[row_idx]
                    formula_type = 'flat_discharge'
            elif 44 <= col_idx <= 47:
                if is_2_to_11_month:
                    cell_value = 充电值_df[time_col].iloc[row_idx]
                    formula_type = 'charge'
                else:
                    cell_value = 放电值_df[time_col].iloc[row_idx]
                    formula_type = 'flat_discharge'
            elif 48 <= col_idx <= 59:
                cell_value = 充电值_df[time_col].iloc[row_idx]
                formula_type = 'charge'

        new_df.at[row_idx, time_col] = cell_value
        formula_type_map.at[row_idx, time_col] = formula_type

# 统计数据计算
低谷时段_cols = [t for t in all_time_cols if t >= "00:00:00" and t <= "05:45:00"]
高峰放电时段_cols = [t for t in all_time_cols if t >= "06:00:00" and t <= "07:45:00"]
平段放电_1月12月_cols = [t for t in all_time_cols if t >= "06:00:00" and t <= "11:45:00"]
平段放电_2_6_10_11月_cols = [t for t in all_time_cols if t >= "08:00:00" and t <= "10:45:00"]
平段放电_7_9月_cols = [t for t in all_time_cols if t >= "09:00:00" and t <= "10:45:00"]
第二次_低谷_1月12月_cols = [t for t in all_time_cols if t >= "12:00:00" and t <= "13:45:00"]
第二次_低谷_2_6_10_11月_cols = [t for t in all_time_cols if t >= "11:00:00" and t <= "13:45:00"]
第二次_低谷_7_9月_cols = [t for t in all_time_cols if t >= "11:00:00" and t <= "12:45:00"]
第二次_平段_1月12月_cols = [t for t in all_time_cols if t >= "14:00:00" and t <= "14:45:00"]
第二次_平段_2_6_10_11月_cols = [t for t in all_time_cols if t >= "14:00:00" and t <= "15:45:00"]
第二次_平段_7_9月_cols = [t for t in all_time_cols if t >= "13:00:00" and t <= "15:45:00"]
第二次_尖峰_1月_12月下旬_cols = [t for t in all_time_cols if t >= "19:00:00" and t <= "20:45:00"]
第二次_尖峰_7_8月_cols = [t for t in all_time_cols if t >= "20:00:00" and t <= "21:45:00"]
第二次_高峰_1月_12月16_31_cols_part1 = [t for t in all_time_cols if t >= "15:00:00" and t <= "18:45:00"]
第二次_高峰_1月_12月16_31_cols_part2 = [t for t in all_time_cols if t >= "21:00:00" and t <= "22:45:00"]
第二次_高峰_2_7月14_cols = [t for t in all_time_cols if t >= "16:00:00" and t <= "21:45:00"]
第二次_高峰_7_8月_part1 = [t for t in all_time_cols if t >= "16:00:00" and t <= "19:45:00"]
第二次_高峰_7_8月_part2 = [t for t in all_time_cols if t >= "22:00:00" and t <= "23:45:00"]
第二次_高峰_9月_cols = [t for t in all_time_cols if t >= "16:00:00" and t <= "23:45:00"]
第二次_高峰_10_11月_cols = [t for t in all_time_cols if t >= "16:00:00" and t <= "21:45:00"]
第二次_高峰_12月1_15_cols = [t for t in all_time_cols if t >= "15:00:00" and t <= "22:45:00"]

stat_data_list = []
上一日循环2剩余 = 0.0
daily_detail_list = []
初始充电容量固定值 = df_pivot.iloc[0]["充电容量"]

for row_idx in range(len(new_df)):
    current_md = md_series.iloc[row_idx]
    current_month = month_series.iloc[row_idx]
    当日充电容量 = df_pivot["充电容量"].iloc[row_idx]
    上午充放容量 = df_pivot["1充放电容量（上午）"].iloc[row_idx]
    下午充放容量 = df_pivot["1充放电容量（下午）"].iloc[row_idx]
    系统留存 = df_pivot["系统留存"].iloc[row_idx]

    第一次_理论_低谷 = min(new_df[低谷时段_cols].iloc[row_idx].sum() / 4, 当日充电容量)
    第一次_理论_平段充电 = 0
    第一次_理论_尖峰放电 = 0

    if ("02-01" <= current_md <= "06-30") or ("10-01" <= current_md <= "11-30"):
        第一次_理论_高峰放电 = min(new_df[高峰放电时段_cols].iloc[row_idx].sum() / 4, 当日充电容量)
    else:
        第一次_理论_高峰放电 = 0

    if "01-01" <= current_md <= "01-31" or "12-01" <= current_md <= "12-31":
        平段计算时段 = 平段放电_1月12月_cols
    elif "02-01" <= current_md <= "06-30" or "10-01" <= current_md <= "11-30":
        平段计算时段 = 平段放电_2_6_10_11月_cols
    elif "07-01" <= current_md <= "09-30":
        平段计算时段 = 平段放电_7_9月_cols
    else:
        平段计算时段 = []

    第一次_理论_平段放电 = min(new_df[平段计算时段].iloc[row_idx].sum() / 4, 当日充电容量) if 平段计算时段 else 0
    第一次理论数据 = [第一次_理论_低谷, 第一次_理论_平段充电, 第一次_理论_尖峰放电, 第一次_理论_高峰放电, 第一次_理论_平段放电]

    if "01-01" <= current_md <= "01-31" or "12-01" <= current_md <= "12-31":
        第二次_低谷计算时段 = 第二次_低谷_1月12月_cols
    elif "02-01" <= current_md <= "06-30" or "10-01" <= current_md <= "11-30":
        第二次_低谷计算时段 = 第二次_低谷_2_6_10_11月_cols
    elif "07-01" <= current_md <= "09-30":
        第二次_低谷计算时段 = 第二次_低谷_7_9月_cols
    else:
        第二次_低谷计算时段 = []

    第二次_理论_低谷 = min(new_df[第二次_低谷计算时段].iloc[row_idx].sum() / 4, 当日充电容量) if 第二次_低谷计算时段 else np.nan

    if "01-01" <= current_md <= "01-31" or "12-01" <= current_md <= "12-31":
        第二次_平段计算时段 = 第二次_平段_1月12月_cols
    elif "02-01" <= current_md <= "06-30" or "10-01" <= current_md <= "11-30":
        第二次_平段计算时段 = 第二次_平段_2_6_10_11月_cols
    elif "07-01" <= current_md <= "09-30":
        第二次_平段计算时段 = 第二次_平段_7_9月_cols
    else:
        第二次_平段计算时段 = []

    if 第二次_平段计算时段:
        第二次_平段时段总和 = new_df[第二次_平段计算时段].iloc[row_idx].sum()
        第二次_理论_平段充电 = min( (第二次_平段时段总和 / 4) * 下午平段充电倍率, 当日充电容量 )
    else:
        第二次_理论_平段充电 = np.nan

    if "01-01" <= current_md <= "01-31":
        第二次_尖峰计算时段 = 第二次_尖峰_1月_12月下旬_cols
        第二次_理论_尖峰放电 = min(new_df[第二次_尖峰计算时段].iloc[row_idx].sum() / 4, 当日充电容量)
    elif "02-01" <= current_md <= "07-14":
        第二次_理论_尖峰放电 = 0
    elif "07-15" <= current_md <= "08-31":
        第二次_尖峰计算时段 = 第二次_尖峰_7_8月_cols
        第二次_理论_尖峰放电 = min(new_df[第二次_尖峰计算时段].iloc[row_idx].sum() / 4, 当日充电容量)
    elif "09-01" <= current_md <= "12-15":
        第二次_理论_尖峰放电 = 0
    elif "12-16" <= current_md <= "12-31":
        第二次_尖峰计算时段 = 第二次_尖峰_1月_12月下旬_cols
        第二次_理论_尖峰放电 = min(new_df[第二次_尖峰计算时段].iloc[row_idx].sum() / 4, 当日充电容量)
    else:
        第二次_理论_尖峰放电 = np.nan

    if "01-01" <= current_md <= "01-31":
        sum1 = new_df[第二次_高峰_1月_12月16_31_cols_part1].iloc[row_idx].sum()
        sum2 = new_df[第二次_高峰_1月_12月16_31_cols_part2].iloc[row_idx].sum()
        第二次_理论_高峰放电 = min( (sum1 + sum2) / 4, 当日充电容量 )
    elif "02-01" <= current_md <= "07-14":
        第二次_理论_高峰放电 = min( new_df[第二次_高峰_2_7月14_cols].iloc[row_idx].sum() / 4, 当日充电容量 )
    elif "07-15" <= current_md <= "08-31":
        sum1 = new_df[第二次_高峰_7_8月_part1].iloc[row_idx].sum()
        sum2 = new_df[第二次_高峰_7_8月_part2].iloc[row_idx].sum()
        第二次_理论_高峰放电 = min( (sum1 + sum2) / 4, 当日充电容量 )
    elif "09-01" <= current_md <= "09-30":
        第二次_理论_高峰放电 = min( new_df[第二次_高峰_9月_cols].iloc[row_idx].sum() / 4, 当日充电容量 )
    elif "10-01" <= current_md <= "11-30":
        第二次_理论_高峰放电 = min( new_df[第二次_高峰_10_11月_cols].iloc[row_idx].sum() / 4, 当日充电容量 )
    elif "12-01" <= current_md <= "12-15":
        第二次_理论_高峰放电 = min( new_df[第二次_高峰_12月1_15_cols].iloc[row_idx].sum() / 4, 当日充电容量 )
    elif "12-16" <= current_md <= "12-31":
        sum1 = new_df[第二次_高峰_1月_12月16_31_cols_part1].iloc[row_idx].sum()
        sum2 = new_df[第二次_高峰_1月_12月16_31_cols_part2].iloc[row_idx].sum()
        第二次_理论_高峰放电 = min( (sum1 + sum2) / 4, 当日充电容量 )
    else:
        第二次_理论_高峰放电 = np.nan

    第二次_理论_平段放电 = 0
    第二次理论数据 = [第二次_理论_低谷, 第二次_理论_平段充电, 第二次_理论_尖峰放电, 第二次_理论_高峰放电, 第二次_理论_平段放电]

    总充电_各项和 = (第一次_理论_低谷 + 第一次_理论_平段充电 + (第二次_理论_低谷 if pd.notna(第二次_理论_低谷) else 0) + (第二次_理论_平段充电 if pd.notna(第二次_理论_平段充电) else 0))
    理论总可充电量 = min(总充电_各项和, 当日充电容量 * 2)
    总放电_各项和 = (第一次_理论_尖峰放电 + 第一次_理论_高峰放电 + 第一次_理论_平段放电 + (第二次_理论_尖峰放电 if pd.notna(第二次_理论_尖峰放电) else 0) + (第二次_理论_高峰放电 if pd.notna(第二次_理论_高峰放电) else 0) + 第二次_理论_平段放电)
    理论总可放电量 = min(总放电_各项和, 当日充电容量 * 2)
    theory_total_data = [理论总可充电量, 理论总可放电量]
    full_theory_data = 第一次理论数据 + 第二次理论数据 + theory_total_data

    if row_idx == 0:
        第一次_实际_低谷 = min(第一次_理论_低谷, 当日充电容量)
        第一次_可用平段容量 = 当日充电容量 - 第一次_实际_低谷
        第一次_实际_平段充电 = min(第一次_理论_平段充电, 第一次_可用平段容量)
        第一次_总充电 = 第一次_实际_低谷 + 第一次_实际_平段充电
        第一次_可放电总上限 = min(第一次_总充电, 上午充放容量) - 系统留存

        第一次_实际_高峰放电 = min(第一次_可放电总上限, 第一次_理论_高峰放电)
        第一次_尖峰可用 = 第一次_可放电总上限 - 第一次_实际_高峰放电
        第一次_实际_尖峰放电 = min(第一次_尖峰可用, 第一次_理论_尖峰放电)
        第一次_平段可用 = 第一次_可放电总上限 - 第一次_实际_高峰放电 - 第一次_实际_尖峰放电
        第一次_实际_平段放电 = min(第一次_平段可用, 第一次_理论_平段放电)

        第一次_总放电 = 第一次_实际_高峰放电 + 第一次_实际_尖峰放电 + 第一次_实际_平段放电
        循环1剩余 = 第一次_总充电 - 第一次_总放电

        第一次实际数据 = [第一次_实际_低谷, 第一次_实际_平段充电, 第一次_实际_尖峰放电, 第一次_实际_高峰放电, 第一次_实际_平段放电, 循环1剩余]
    else:
        第一次_实际_低谷 = min(第一次_理论_低谷, 当日充电容量 - 上一日循环2剩余)
        第一次_可用平段容量 = 当日充电容量 - 第一次_实际_低谷 - 上一日循环2剩余
        第一次_实际_平段充电 = min(第一次_理论_平段充电, 第一次_可用平段容量)
        第一次_总充电 = 第一次_实际_低谷 + 第一次_实际_平段充电 + 上一日循环2剩余
        第一次_可放电总上限 = min(第一次_总充电, 上午充放容量) - 系统留存

        第一次_实际_高峰放电 = min(第一次_可放电总上限, 第一次_理论_高峰放电)
        第一次_尖峰可用 = 第一次_可放电总上限 - 第一次_实际_高峰放电
        第一次_实际_尖峰放电 = min(第一次_尖峰可用, 第一次_理论_尖峰放电)
        第一次_平段可用 = 第一次_可放电总上限 - 第一次_实际_高峰放电 - 第一次_实际_尖峰放电
        第一次_实际_平段放电 = min(第一次_平段可用, 第一次_理论_平段放电)

        第一次_总放电 = 第一次_实际_高峰放电 + 第一次_实际_尖峰放电 + 第一次_实际_平段放电
        循环1剩余 = 第一次_总充电 - 第一次_总放电

        第一次实际数据 = [第一次_实际_低谷, 第一次_实际_平段充电, 第一次_实际_尖峰放电, 第一次_实际_高峰放电, 第一次_实际_平段放电, 循环1剩余]

    第二次_可用低谷容量 = 当日充电容量 - 循环1剩余
    第二次_实际_低谷 = min(第二次_理论_低谷 if pd.notna(第二次_理论_低谷) else 0, 第二次_可用低谷容量)
    第二次_可用平段容量 = 第二次_可用低谷容量 - 第二次_实际_低谷
    第二次_实际_平段充电 = min(第二次_理论_平段充电 if pd.notna(第二次_理论_平段充电) else 0, 第二次_可用平段容量)
    第二次_总充电后电量 = 循环1剩余 + 第二次_实际_低谷 + 第二次_实际_平段充电
    第二次_可放电总上限 = min(第二次_总充电后电量, 下午充放容量) - 系统留存
    第二次_实际_尖峰放电 = min(第二次_可放电总上限, 第二次_理论_尖峰放电 if pd.notna(第二次_理论_尖峰放电) else 0)
    第二次_高峰可用 = 第二次_可放电总上限 - 第二次_实际_尖峰放电
    第二次_实际_高峰放电 = min(第二次_高峰可用, 第二次_理论_高峰放电 if pd.notna(第二次_理论_高峰放电) else 0)
    第二次_平段可用 = 第二次_可放电总上限 - 第二次_实际_尖峰放电 - 第二次_实际_高峰放电
    第二次_实际_平段放电 = min(第二次_平段可用, 第二次_理论_平段放电)
    第二次_总放电 = 第二次_实际_尖峰放电 + 第二次_实际_高峰放电 + 第二次_实际_平段放电
    循环2剩余 = 第二次_总充电后电量 - 第二次_总放电

    第二次实际数据 = [第二次_实际_低谷, 第二次_实际_平段充电, 第二次_实际_尖峰放电, 第二次_实际_高峰放电, 第二次_实际_平段放电, 循环2剩余]

    循环1 = 第一次实际数据[2] + 第一次实际数据[3] + 第一次实际数据[4]
    循环2 = 第二次实际数据[2] + 第二次实际数据[3] + 第二次实际数据[4]

    if 充放模式 == 2:
        折算天数 = (循环1 + 循环2) / 当日充电容量 / 2
    elif 充放模式 == 1:
        折算天数 = (循环1 + 循环2) / 当日充电容量
    else:
        折算天数 = np.nan

    循环1计数 = 循环1 / 当日充电容量
    循环2计数 = 循环2 / 当日充电容量
    低谷充电 = 第一次实际数据[0] + 第二次实际数据[0]
    平段充电 = 第一次实际数据[1] + 第二次实际数据[1]
    尖峰放电 = 第一次实际数据[2] + 第二次实际数据[2]
    高峰放电 = 第一次实际数据[3] + 第二次实际数据[3]
    平段放电 = 第一次实际数据[4] + 第二次实际数据[4]
    总放电 = 尖峰放电 + 高峰放电 + 平段放电
    满容量天数 = 总放电 / 初始充电容量固定值 / 2

    summary_data = [循环1, 循环2, 折算天数, 循环1计数, 循环2计数, 低谷充电, 平段充电, 尖峰放电, 高峰放电, 平段放电, 总放电, 满容量天数]

    daily_detail_list.append({
        "月份": current_month,
        "充电容量": 当日充电容量,
        "理论总可充电量": 理论总可充电量,
        "理论总可放电量": 理论总可放电量,
        "折算天数": 折算天数,
        "满容量天数": 满容量天数,
        "低谷充电": 低谷充电,
        "平段充电": 平段充电,
        "尖峰放电": 尖峰放电,
        "高峰放电": 高峰放电,
        "平段放电": 平段放电,
        "总放电": 总放电
    })

    上一日循环2剩余 = 循环2剩余
    full_actual_data = 第一次实际数据 + 第二次实际数据
    当日全量统计数据 = full_theory_data + full_actual_data + summary_data
    stat_data_list.append(当日全量统计数据)

# 月度汇总
df_daily = pd.DataFrame(daily_detail_list)
总计_折算天数 = df_daily["折算天数"].sum()
总计_满容量天数 = df_daily["满容量天数"].sum()

df_monthly_agg = df_daily.groupby("月份").agg(
    天数=("满容量天数", "sum"),
    折算天数=("折算天数", "sum"),
    谷充电量=("低谷充电", "sum"),
    平充电量=("平段充电", "sum"),
    尖放电量=("尖峰放电", "sum"),
    峰放电量=("高峰放电", "sum"),
    平放电量=("平段放电", "sum"),
    总理论可充电=("理论总可充电量", "sum"),
    总理论可放电=("理论总可放电量", "sum"),
    总充电容量2倍=("充电容量", lambda x: (x*2).sum())
).reset_index()

month_days_map = {1:31, 2:28, 3:31, 4:30, 5:31, 6:30, 7:31, 8:31, 9:30, 10:31, 11:30, 12:31}
month_type_map = {1:"特殊", 2:"正常", 3:"正常", 4:"正常", 5:"正常", 6:"正常", 7:"特殊", 8:"特殊", 9:"正常", 10:"正常", 11:"正常", 12:"特殊"}

monthly_summary_list = []
for month in range(1, 13):
    if month in df_monthly_agg["月份"].values:
        month_agg = df_monthly_agg[df_monthly_agg["月份"] == month].iloc[0]
    else:
        month_agg = pd.Series(0, index=df_monthly_agg.columns)

    分类 = month_type_map[month]
    每月天数 = month_days_map[month]
    谷充电量 = month_agg["谷充电量"]
    平充电量 = month_agg["平充电量"]
    总充电量 = 谷充电量 + 平充电量
    尖放电量 = month_agg["尖放电量"]
    峰放电量 = month_agg["峰放电量"]
    平放电量 = month_agg["平放电量"]
    总放电量 = 尖放电量 + 峰放电量 + 平放电量

    谷充电占比 = 谷充电量 / 总充电量 if 总充电量 != 0 else 0
    平充电占比 = 平充电量 / 总充电量 if 总充电量 != 0 else 0
    尖放电占比 = 尖放电量 / 总放电量 if 总放电量 != 0 else 0
    峰放电占比 = 峰放电量 / 总放电量 if 总放电量 != 0 else 0
    平放电占比 = 平放电量 / 总放电量 if 总放电量 != 0 else 0

    总理论可充电 = month_agg["总理论可充电"]
    总理论可放电 = month_agg["总理论可放电"]
    总充电容量2倍 = month_agg["总充电容量2倍"]
    理论可充电情况 = 总理论可充电 / 总充电容量2倍 if 总充电容量2倍 != 0 else 0
    理论可放电情况 = 总理论可放电 / 总充电容量2倍 if 总充电容量2倍 != 0 else 0
    综合充放电数值 = min(理论可充电情况, 理论可放电情况)

    充放差值 = 理论可充电情况 - 理论可放电情况
    if 充放差值 >= 0.05:
        综合充放电文本 = "消纳不足"
    elif 充放差值 <= -0.05:
        综合充放电文本 = "充电不足"
    else:
        综合充放电文本 = "充放平衡"

    monthly_summary_list.append({
        "月份": month,
        "分类": 分类,
        "天数": month_agg["天数"],
        "折算天数": month_agg["折算天数"],
        "谷充电量(度)": 谷充电量,
        "平充电量(度)": 平充电量,
        "总充电量(度)": 总充电量,
        "尖放电量(度)": 尖放电量,
        "峰放电量(度)": 峰放电量,
        "平放电量(度)": 平放电量,
        "总放电量(度)": 总放电量,
        "谷充电占比": 谷充电占比,
        "平充电占比": 平充电占比,
        "尖放电占比": 尖放电占比,
        "峰放电占比": 峰放电占比,
        "平放电占比": 平放电占比,
        "每月天数": 每月天数,
        "理论可充电情况": 理论可充电情况,
        "理论可放电情况": 理论可放电情况,
        "综合充放电数值": 综合充放电数值,
        "综合充放电文本": 综合充放电文本
    })

总计_天数 = df_daily["满容量天数"].sum()
总计_折算天数 = df_daily["折算天数"].sum()
总计_谷充电 = df_daily["低谷充电"].sum()
总计_平充电 = df_daily["平段充电"].sum()
总计_总充电 = 总计_谷充电 + 总计_平充电
总计_尖放电 = df_daily["尖峰放电"].sum()
总计_峰放电 = df_daily["高峰放电"].sum()
总计_平放电 = df_daily["平段放电"].sum()
总计_总放电 = 总计_尖放电 + 总计_峰放电 + 总计_平放电

总计_谷充占比 = 总计_谷充电 / 总计_总充电 if 总计_总充电 !=0 else 0
总计_平充占比 = 总计_平充电 / 总计_总充电 if 总计_总充电 !=0 else 0
总计_尖放占比 = 总计_尖放电 / 总计_总放电 if 总计_总放电 !=0 else 0
总计_峰放占比 = 总计_峰放电 / 总计_总放电 if 总计_总放电 !=0 else 0
总计_平放占比 = 总计_平放电 / 总计_总放电 if 总计_总放电 !=0 else 0

总计_总理论充 = df_daily["理论总可充电量"].sum()
总计_总理论放 = df_daily["理论总可放电量"].sum()
总计_总充容量2倍 = (df_daily["充电容量"]*2).sum()
总计_理论充情况 = 总计_总理论充 / 总计_总充容量2倍 if 总计_总充容量2倍 !=0 else 0
总计_理论放情况 = 总计_总理论放 / 总计_总充容量2倍 if 总计_总充容量2倍 !=0 else 0
总计_综合数值 = min(总计_理论充情况, 总计_理论放情况)

monthly_summary_list.append({
    "月份": "总计",
    "分类": "",
    "天数": 总计_天数,
    "折算天数": 总计_折算天数,
    "谷充电量(度)": 总计_谷充电,
    "平充电量(度)": 总计_平充电,
    "总充电量(度)": 总计_总充电,
    "尖放电量(度)": 总计_尖放电,
    "峰放电量(度)": 总计_峰放电,
    "平放电量(度)": 总计_平放电,
    "总放电量(度)": 总计_总放电,
    "谷充电占比": 总计_谷充占比,
    "平充电占比": 总计_平充占比,
    "尖放电占比": 总计_尖放占比,
    "峰放电占比": 总计_峰放占比,
    "平放电占比": 总计_平放占比,
    "每月天数": 365,
    "理论可充电情况": 总计_理论充情况,
    "理论可放电情况": 总计_理论放情况,
    "综合充放电数值": 总计_综合数值,
    "综合充放电文本": ""
})

df_monthly_summary = pd.DataFrame(monthly_summary_list)

# 生成回收期表, 并获取实际回收期数值
payback_df, global_actual_final = generate_payback_table()
actual_final_str = f"{global_actual_final:.2f}" if global_actual_final is not None else "无法回收"


# ==============================================
# 9. 统一输出到Excel
# ==============================================
def create_sensitivity_analysis_sheet(writer, sheet_name):
    wb = writer.book
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)

    # 样式
    _thin_border = Border(
        left=Side(style='thin'), right=Side(style='thin'),
        top=Side(style='thin'), bottom=Side(style='thin')
    )
    _thick_border = Border(
        left=Side(style='medium'), right=Side(style='medium'),
        top=Side(style='medium'), bottom=Side(style='medium')
    )
    _center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    _header_font = Font(name='微软雅黑', size=10, bold=True)
    _data_font = Font(name='微软雅黑', size=9)
    _yellow_fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
    _light_yellow_fill = PatternFill(start_color='FFFF99', end_color='FFFF99', fill_type='solid')

    # 1. 顶部大标题
    ws.merge_cells('A1:F1')
    cell_title = ws['A1']
    cell_title.value = "安徽"
    cell_title.font = Font(name='微软雅黑', size=16, bold=True)
    cell_title.alignment = _center
    cell_title.border = _thick_border

    # 2. 参数行
    设备低压成本 = 设备成本_万_台 / 系统额定电量_kWh * 10
    param_text = (
        f"充电电价:{充电含税电价:.2f}\n"
        f"放电电价:{放电含税电价:.2f}\n"
        f"价差:{(放电含税电价 - 充电含税电价):.2f}\n"
        f"设备低压成本:{设备低压成本:.2f}"
    )
    ws.merge_cells('A2:F2')
    cell_param = ws['A2']
    cell_param.value = param_text
    cell_param.fill = _yellow_fill
    cell_param.font = Font(name='微软雅黑', size=12, bold=True)
    cell_param.alignment = _center
    cell_param.border = _thick_border
    ws.row_dimensions[2].height = 80

    # 3. 列表头
    headers = ["运行天数", "折扣", "居间", "施工", f"{selected_model}设备", "回收周期"]
    for col_num, header in enumerate(headers, 1):
        cell = ws.cell(row=3, column=col_num, value=header)
        cell.font = _header_font
        cell.fill = _yellow_fill
        cell.alignment = _center
        cell.border = _thick_border

    # 4. 定义排列组合
    discount_list = [0.85, 0.8, 0.75]
    # 使用你提供的居间费率列表，并确保与截图一致
    jujian_list = [0.12, 0.1, 0.08, 0.06, 0.04] # 确保与实际逻辑保持一致
    jujian_count_per_discount = {
        0.85: [0.12, 0.1, 0.08, 0.06], # 4个居间费
        0.8: [0.12, 0.1, 0.08, 0.06],  # 4个居间费
        0.75: [0.1, 0.08, 0.06, 0.04]   # 4个居间费 (根据截图，0.75折扣下没有0.12，第一个是0.1)
    }


    # 5. 填充数据
    current_data_row = 4

    for disc_idx, disc in enumerate(discount_list):
        start_row_group = current_data_row

        # 获取当前折扣下对应的居间列表
        current_jujians = jujian_count_per_discount[disc]
        jj_count = len(current_jujians)
        results_in_group = []

        # 合并：运行天数
        ws.merge_cells(start_row=current_data_row, start_column=1,
                      end_row=current_data_row + jj_count - 1, end_column=1)
        c1 = ws.cell(row=current_data_row, column=1, value=round(总计_折算天数, 2))
        c1.alignment = _center
        c1.border = _thin_border
        c1.font = _data_font

        # 合并：折扣
        ws.merge_cells(start_row=current_data_row, start_column=2,
                      end_row=current_data_row + jj_count - 1, end_column=2)
        c2 = ws.cell(row=current_data_row, column=2, value=disc)
        c2.alignment = _center
        c2.border = _thin_border
        c2.font = _data_font

        # 合并：施工
        ws.merge_cells(start_row=current_data_row, start_column=4,
                      end_row=current_data_row + jj_count - 1, end_column=4)
        施工单位成本_万元每度 = round(施工费_万_台 * 台数 / 系统额定电量_kWh, 2) if 系统额定电量_kWh else 0
        c4 = ws.cell(row=current_data_row, column=4, value=f"{施工单位成本_万元每度:.2f}（{施工费_万_台}万/台）")
        c4.alignment = _center
        c4.border = _thin_border
        c4.font = _data_font

        # 合并：设备
        ws.merge_cells(start_row=current_data_row, start_column=5,
                      end_row=current_data_row + jj_count - 1, end_column=5)
        设备单位成本_万元每度 = round(设备成本_万_台 * 台数 / 系统额定电量_kWh, 2) if 系统额定电量_kWh else 0
        c5 = ws.cell(row=current_data_row, column=5, value=f"{设备单位成本_万元每度:.2f}（{设备成本_万_台}万/台）")
        c5.alignment = _center
        c5.border = _thin_border
        c5.font = _data_font

        # 填写居间和计算回收期
        for idx in range(jj_count):
            jj = current_jujians[idx] # 使用当前折扣下的居间费率
            r = current_data_row + idx

            c_jj = ws.cell(row=r, column=3, value=jj)
            c_jj.alignment = _center
            c_jj.border = _thin_border
            c_jj.font = _data_font

            pb_val = calculate_payback_for_combination(disc, jj)

            c_pb = ws.cell(row=r, column=6, value=pb_val)
            c_pb.alignment = _center
            c_pb.border = _thin_border
            c_pb.font = _data_font
            c_pb.number_format = '0.00'

            results_in_group.append((r, pb_val))

        # 找出该折扣下的最小值并标黄
        if results_in_group:
            valid_results = [t for t in results_in_group if t[1] is not None and not pd.isna(t[1])]
            if valid_results:
                min_row, min_val = min(valid_results, key=lambda x: x[1])
                ws.cell(row=min_row, column=6).fill = _light_yellow_fill

        current_data_row += jj_count

    # 设置列宽
    ws.column_dimensions['A'].width = 12
    ws.column_dimensions['B'].width = 8
    ws.column_dimensions['C'].width = 8
    ws.column_dimensions['D'].width = 20
    ws.column_dimensions['E'].width = 22
    ws.column_dimensions['F'].width = 12

    # 设置边框
    for row in range(1, current_data_row):
        for col in range(1, 7):
            cell = ws.cell(row=row, column=col)
            if cell.border is None or cell.border.left.style is None: # 避免重复设置已合并单元格的边框
                cell.border = _thin_border

with pd.ExcelWriter("表2数据（安徽）.xlsx", engine="openpyxl") as writer:
    wb = writer.book
    if "Sheet" in wb.sheetnames:
        del wb["Sheet"]

    # 1. 容量-消纳测算
    sheet1 = wb.create_sheet("容量-消纳测算", 0)
    sheet1["A1"] = "池州丰林木业有限公司"
    sheet1.merge_cells(start_row=1, start_column=1, end_row=1, end_column=3)
    title_cell = sheet1["A1"]
    title_cell.font = title_font
    title_cell.alignment = center_align
    title_cell.border = thin_border

    df_pivot.to_excel(writer, sheet_name="容量-消纳测算", startrow=1, index=False, header=True)

    original_last_row = sheet1.max_row
    stat_table_total_start_col = len(df_pivot.columns) + 2
    new_table_start_row = original_last_row + 4
    data_start_row = new_table_start_row + 1

    theory_header_level1 = "各时段理论可充放电量"
    theory_header_level2 = ["第一次充放", "第二次充放"]
    theory_header_level3 = ["低谷可充电量", "平段可充电量", "尖峰可放电量", "高峰可放电量", "平段可放电量", "低谷可充电量", "平段可充电量", "尖峰可放电量", "高峰可放电量", "平段可放电量"]
    theory_total_headers = ["理论总可充电量", "理论总可放电量"]

    actual_header_level1 = "各时段实际可充放电量"
    actual_header_level2 = ["第一次充放", "第二次充放"]
    actual_header_level3 = ["低谷充电量", "平段充电量", "尖峰放电量", "高峰放电量", "平段放电量", "循环1剩余", "低谷充电量", "平段充电量", "尖峰放电量", "高峰放电量", "平段放电量", "循环2剩余"]

    summary_headers = ["循环1", "循环2", "折算天数", "循环1计数", "循环2计数", "低谷充电", "平段充电", "尖峰放电", "高峰放电", "平段放电", "总放电", "满容量天数"]

    for col_idx, col_name in enumerate(df_pivot.columns, start=1):
        sheet1.cell(row=new_table_start_row, column=col_idx, value=col_name)
    for row_idx, row_data in new_df.iterrows():
        excel_row = data_start_row + row_idx
        for col_idx, col_name in enumerate(df_pivot.columns, start=1):
            value = row_data[col_name]
            if pd.notna(value):
                sheet1.cell(row=excel_row, column=col_idx, value=value)

    sheet1.merge_cells(start_row=original_last_row + 2, start_column=stat_table_total_start_col, end_row=original_last_row + 2, end_column=stat_table_total_start_col + 9)
    theory_title_cell = sheet1.cell(row=original_last_row + 2, column=stat_table_total_start_col, value=theory_header_level1)
    theory_title_cell.fill = fill_header_yellow
    theory_title_cell.font = header_font
    theory_title_cell.alignment = center_align
    theory_title_cell.border = thin_border

    sheet1.merge_cells(start_row=original_last_row + 3, start_column=stat_table_total_start_col, end_row=original_last_row + 3, end_column=stat_table_total_start_col + 4)
    theory_first_cell = sheet1.cell(row=original_last_row + 3, column=stat_table_total_start_col, value=theory_header_level2[0])
    theory_first_cell.fill = fill_header_yellow
    theory_first_cell.font = header_font
    theory_first_cell.alignment = center_align
    theory_first_cell.border = thin_border

    sheet1.merge_cells(start_row=original_last_row + 3, start_column=stat_table_total_start_col + 5, end_row=original_last_row + 3, end_column=stat_table_total_start_col + 9)
    theory_second_cell = sheet1.cell(row=original_last_row + 3, column=stat_table_total_start_col + 5, value=theory_header_level2[1])
    theory_second_cell.fill = fill_header_yellow
    theory_second_cell.font = header_font
    theory_second_cell.alignment = center_align
    theory_second_cell.border = thin_border

    for col_idx, header_name in enumerate(theory_header_level3):
        cell = sheet1.cell(row=new_table_start_row, column=stat_table_total_start_col + col_idx, value=header_name)
        cell.fill = fill_header_yellow
        cell.font = header_font
        cell.alignment = center_align
        cell.border = thin_border

    for col_idx, header_name in enumerate(theory_total_headers):
        cell = sheet1.cell(row=new_table_start_row, column=stat_table_total_start_col + 10 + col_idx, value=header_name)
        cell.fill = fill_header_yellow
        cell.font = header_font
        cell.alignment = center_align
        cell.border = thin_border

    actual_table_start_col = stat_table_total_start_col + 12
    sheet1.merge_cells(start_row=original_last_row + 2, start_column=actual_table_start_col, end_row=original_last_row + 2, end_column=actual_table_start_col + 11)
    actual_title_cell = sheet1.cell(row=original_last_row + 2, column=actual_table_start_col, value=actual_header_level1)
    actual_title_cell.fill = fill_header_yellow
    actual_title_cell.font = header_font
    actual_title_cell.alignment = center_align
    actual_title_cell.border = thin_border

    sheet1.merge_cells(start_row=original_last_row + 3, start_column=actual_table_start_col, end_row=original_last_row + 3, end_column=actual_table_start_col + 5)
    actual_first_cell = sheet1.cell(row=original_last_row + 3, column=actual_table_start_col, value=actual_header_level2[0])
    actual_first_cell.fill = fill_header_yellow
    actual_first_cell.font = header_font
    actual_first_cell.alignment = center_align
    actual_first_cell.border = thin_border

    sheet1.merge_cells(start_row=original_last_row + 3, start_column=actual_table_start_col + 6, end_row=original_last_row + 3, end_column=actual_table_start_col + 11)
    actual_second_cell = sheet1.cell(row=original_last_row + 3, column=actual_table_start_col + 6, value=actual_header_level2[1])
    actual_second_cell.fill = fill_header_yellow
    actual_second_cell.font = header_font
    actual_second_cell.alignment = center_align
    actual_second_cell.border = thin_border

    for col_idx, header_name in enumerate(actual_header_level3):
        cell = sheet1.cell(row=new_table_start_row, column=actual_table_start_col + col_idx, value=header_name)
        cell.fill = fill_header_yellow
        cell.font = header_font
        cell.alignment = center_align
        cell.border = thin_border

    summary_table_start_col = actual_table_start_col + 12
    for col_idx, header_name in enumerate(summary_headers):
        cell = sheet1.cell(row=new_table_start_row, column=summary_table_start_col + col_idx, value=header_name)
        cell.fill = fill_header_yellow
        cell.font = header_font
        cell.alignment = center_align
        cell.border = thin_border

    for row_idx, 当日统计数据 in enumerate(stat_data_list):
        excel_row = data_start_row + row_idx
        for col_idx, value in enumerate(当日统计数据):
            if pd.notna(value):
                sheet1.cell(row=excel_row, column=stat_table_total_start_col + col_idx, value=value)

    max_row_sheet1 = sheet1.max_row
    max_col_sheet1 = sheet1.max_column

    for col_idx in range(1, max_col_sheet1 + 1):
        col_letter = get_column_letter(col_idx)
        if col_idx == 1:
            sheet1.column_dimensions[col_letter].width = 6
        elif col_idx == 2:
            sheet1.column_dimensions[col_letter].width = 12
        elif 3 <= col_idx <= 98:
            sheet1.column_dimensions[col_letter].width = 8
        else:
            sheet1.column_dimensions[col_letter].width = 11

    for row in range(1, max_row_sheet1 + 1):
        for col in range(1, max_col_sheet1 + 1):
            cell = sheet1.cell(row=row, column=col)
            if cell.value is None and row > 1:
                continue
            cell.alignment = center_align
            cell.border = thin_border

            if row == 1:
                continue
            elif row == 2 or row == new_table_start_row or (original_last_row + 2 <= row <= new_table_start_row):
                continue
            else:
                cell.font = data_font
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '0.00'

    time_col_excel_start = 3
    for row_idx in range(len(df_pivot)):
        excel_row = data_start_row + row_idx
        for col_idx, time_col in enumerate(all_time_cols):
            excel_col = time_col_excel_start + col_idx
            cell = sheet1.cell(row=excel_row, column=excel_col)
            current_formula_type = formula_type_map.at[row_idx, time_col]
            if current_formula_type == 'charge':
                cell.fill = fill_charge
            elif current_formula_type == 'flat_discharge':
                cell.fill = fill_flat
            elif current_formula_type == 'peak_discharge':
                cell.fill = fill_peak

    # 2. 容量-储能配置
    sheet_monthly = wb.create_sheet("容量-储能配置", 1)

    sheet_monthly.merge_cells(start_row=1, start_column=1, end_row=1, end_column=15)
    title_cell = sheet_monthly.cell(row=1, column=1, value="安徽池州丰林木业有限公司储能装机容量测算")
    title_cell.fill = PatternFill(start_color='0066CC', end_color='0066CC', fill_type='solid')
    title_cell.font = Font(name='微软雅黑', size=16, bold=True, color='FFFFFF')
    title_cell.alignment = center_align

    header_row1 = [
        "变压器容量\n(kVA)", "安全系数", "功率上限\n(kW)", "基本电费", "需量数据选择", "充放模式", "设备型号", "台数", "系统功率\n(kW)", "系统额定电量\n(KWh)",
        "1、7、8、12\n尖峰前高峰放电倍", "放电保护", "设备规格\n(功率)", "设备规格\n(标定容量)", "设备规格\n(实际容量)"
    ]
    value_row1 = [
        变压器容量_kVA, 安全系数, round(功率上限_kW), "按容", "原始数据", 充放模式, selected_model, 台数, 系统功率_kW, 系统额定电量_kWh,
        尖峰前高峰放电倍率, 系统功率_kW * 0.1, 设备规格_功率_kW, 设备规格_标定容量_kWh, 设备规格_实际容量_kWh
    ]

    for col_idx, header in enumerate(header_row1, start=1):
        cell = sheet_monthly.cell(row=2, column=col_idx, value=header)
        cell.font = header_font
        cell.alignment = center_align
        cell.border = thin_border

    yellow_cols_row1 = [1,4,5,6,7,8,12]
    blue_cols_row1 = [2,11,13,14,15]
    for col_idx, value in enumerate(value_row1, start=1):
        cell = sheet_monthly.cell(row=3, column=col_idx, value=value)
        cell.font = data_font
        cell.alignment = center_align
        cell.border = thin_border
        if col_idx in yellow_cols_row1:
            cell.fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
        elif col_idx in blue_cols_row1:
            cell.fill = PatternFill(start_color='00B0F0', end_color='00B0F0', fill_type='solid')
        if isinstance(value, float) and value < 10:
            cell.number_format = '0.00'

    header_row2 = [
        "临界容量", "余量保护值", "DOD", "年衰减率", "日衰减率", "商务模式", "折扣", "居间", "系统实际电量\n(KWh)",
        "上午平段\n限制放电量", "下午限制放电\n量", "下午平段\n充电倍率", "0.25C有效天数", "2充放回收期", "初始投资不含税"
    ]
    value_row2 = [
        临界容量,178, "100%", "2.50%", "0.007%", 商务模式, 折扣, 居间费率, 系统实际电量_kWh,
        f"{上午平段限制放电量比例*100}%", f"{下午限制放电量比例*100}%", 下午平段充电倍率, round(总计_折算天数, 2), actual_final_str, initial_invest_excl
    ]

    for col_idx, header in enumerate(header_row2, start=1):
        cell = sheet_monthly.cell(row=4, column=col_idx, value=header)
        cell.font = header_font
        cell.alignment = center_align
        cell.border = thin_border

    yellow_cols_row2 = [6,7,8]
    blue_cols_row2 = [4,10,11,12]
    green_cols_row2 = [13,14,15] # 调整索引，因为移除了一个列
    for col_idx, value in enumerate(value_row2, start=1):
        cell = sheet_monthly.cell(row=5, column=col_idx, value=value)
        cell.font = data_font
        cell.alignment = center_align
        cell.border = thin_border
        if col_idx in yellow_cols_row2:
            cell.fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
        elif col_idx in blue_cols_row2:
            cell.fill = PatternFill(start_color='00B0F0', end_color='00B0F0', fill_type='solid')
        elif col_idx in green_cols_row2:
            cell.fill = PatternFill(start_color='92D050', end_color='92D050', fill_type='solid')
        if isinstance(value, float):
            cell.number_format = '0.00'
        elif isinstance(value, str) and value.endswith('%'): # 处理百分比字符串
            cell.number_format = '0.00%' if value != "100%" else '0' # 如果是100%就不显示两位小数

    loss_start_row = 7
    sheet_monthly.merge_cells(start_row=loss_start_row, start_column=1, end_row=loss_start_row, end_column=2)
    loss_title_cell = sheet_monthly.cell(row=loss_start_row, column=1, value="充放10年电池损耗")
    loss_title_cell.font = header_font
    loss_title_cell.alignment = center_align
    loss_title_cell.border = thin_border
    loss_title_cell.fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')

    loss_values = [0.9651, 0.9113, 0.8773, 0.8483, 0.8227, 0.7998, 0.7782, 0.7579, 0.7391, 0.7208]
    for col_offset, val in enumerate(loss_values):
        cell = sheet_monthly.cell(row=loss_start_row, column=3+col_offset, value=val)
        cell.font = data_font
        cell.alignment = center_align
        cell.border = thin_border
        cell.number_format = '0.00%'

    monthly_start_row = loss_start_row + 2
    monthly_header_cols = df_monthly_summary.columns.tolist()

    # 写入表头，处理合并单元格
    for col_idx in range(19): # 前19列是 df_monthly_summary 的列
        header_cell = sheet_monthly.cell(row=monthly_start_row, column=col_idx+1, value=monthly_header_cols[col_idx])
        header_cell.font = header_font
        header_cell.alignment = center_align
        header_cell.border = thin_border
        header_cell.fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid') # 默认黄色表头

    # "综合充放电情况" 合并单元格
    sheet_monthly.merge_cells(start_row=monthly_start_row, start_column=20, end_row=monthly_start_row, end_column=21)
    comp_header_cell = sheet_monthly.cell(row=monthly_start_row, column=20, value="综合充放电情况")
    comp_header_cell.font = header_font
    comp_header_cell.alignment = center_align
    comp_header_cell.border = thin_border
    comp_header_cell.fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid') # 默认黄色表头


    for row_idx in range(len(df_monthly_summary)):
        excel_row = monthly_start_row + 1 + row_idx
        row_data = df_monthly_summary.iloc[row_idx]
        for col_idx in range(len(monthly_header_cols)):
            excel_col = col_idx + 1
            value = row_data[monthly_header_cols[col_idx]]
            sheet_monthly.cell(row=excel_row, column=excel_col, value=value)

    monthly_header_fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
    month_red_fill = PatternFill(start_color='FF0000', end_color='FF0000', fill_type='solid')
    month_white_font = Font(name='微软雅黑', size=10, bold=True, color='FFFFFF')

    max_row_monthly = sheet_monthly.max_row
    max_col_monthly = sheet_monthly.max_column

    # 数据行样式设置
    for row in range(monthly_start_row + 1, max_row_monthly + 1):
        # 特殊月份红底白字设置
        month_cell = sheet_monthly.cell(row=row, column=1)
        month_value = month_cell.value
        if month_value in [1,7,8,12]:
            month_cell.fill = month_red_fill
            month_cell.font = month_white_font

        # 全单元格通用样式
        for col in range(1, max_col_monthly + 1):
            cell = sheet_monthly.cell(row=row, column=col)
            cell.alignment = center_align
            cell.border = thin_border
            if col not in [1,2,21]: # 月份、分类、综合文本列
                cell.font = data_font

            # 数字格式设置
            if col in [3,4,5,6,7,8,9,10,11]: # 天数、电量类，整数格式
                cell.number_format = '0'
            elif col in [12,13,14,15,16,18,19,20]: # 占比、理论情况，百分比格式
                cell.number_format = '0.00%'
            elif col == 17: # 每月天数
                cell.number_format = '0'
            elif col == 21: # 综合充放电文本
                pass # 保持字符串格式

    col_width_map = {
        1:6, 2:6, 3:7, 4:8, 5:11, 6:11, 7:11, 8:11, 9:11, 10:11, 11:11,
        12:9, 13:9, 14:9, 15:9, 16:9, 17:8, 18:13, 19:13, 20:10, 21:10
    }
    for col_idx, width in col_width_map.items():
        col_letter = get_column_letter(col_idx)
        sheet_monthly.column_dimensions[col_letter].width = width

    last_row_monthly = sheet_monthly.max_row
    start_row_vars = last_row_monthly + 2
    var_names = ["加权电价放电电价", "充电电价", "充放差价"]
    var_values = [放电含税电价, 充电含税电价, 放电含税电价 - 充电含税电价] # 使用计算值
    for i, (name, value) in enumerate(zip(var_names, var_values)):
        row = start_row_vars + i
        sheet_monthly.merge_cells(start_row=row, start_column=1, end_row=row, end_column=2)
        cell_name = sheet_monthly.cell(row=row, column=1, value=name)
        cell_name.font = header_font
        cell_name.alignment = center_align
        cell_name.border = thin_border
        cell_val = sheet_monthly.cell(row=row, column=3, value=value)
        cell_val.font = data_font
        cell_val.alignment = center_align
        cell_val.border = thin_border
        cell_val.number_format = '0.0000'

    # 3. 2充放收益测算
    sheet_profit = wb.create_sheet("2充放收益测算", 2)

    discharge_tax = 放电含税电价
    charge_tax = 充电含税电价
    discharge_excl = 放电不含税电价
    charge_excl = 充电不含税电价
    investment_degree = 系统额定电量_kWh
    actual_degree = 系统实际电量_kWh
    # cost_per_kwh 已经在全局派生参数中定义
    # initial_invest_excl 已经在全局派生参数中定义
    initial_invest_tax = initial_invest_excl * 1.13 # 使用全局的 initial_invest_excl
    run_days = 总计_折算天数
    efficiency = 系统效率_收益测算
    om_cost_rate = 运维成本率
    tax_rate = 所得税率
    interest_rate = 年利率
    emc_discount = 电价折扣

    battery_cap = [0.965, 0.911, 0.877, 0.848, 0.823, 0.800, 0.778, 0.758, 0.965, 0.911, 0.877, 0.848, 0.823, 0.800, 0.778, 0.758]
    years = list(range(1, 17))
    epc_annual_profit = []
    npv_prev = 0
    npv_list = []
    for i in range(16):
        charge_vol = actual_degree * run_days / 10000 * 2 * battery_cap[i] / (1 - (0.5 - efficiency/2))
        discharge_vol = charge_vol * efficiency
        income = discharge_excl * discharge_vol
        cost_charge = charge_excl * charge_vol
        om_cost = 0 if i < 2 else 台数 * om_cost_rate * 20
        profit_before_tax = income - cost_charge - om_cost
        profit_tax = profit_before_tax * 1.13
        epc_annual_profit.append(profit_tax)
        if i == 0:
            npv = profit_tax - initial_invest_tax
        else:
            npv = npv_prev + profit_tax
        npv_prev = npv
        npv_list.append(npv)

    epc_16year_total_profit = sum(epc_annual_profit)

    payback_period = ""
    for i in range(len(npv_list)):
        if npv_list[i] >= 0:
            if i == 0:
                payback_period = f"{1}年"
            else:
                prev_npv = npv_list[i-1]
                curr_profit = epc_annual_profit[i]
                if curr_profit != 0: # 避免除以零
                    payback = i + (abs(prev_npv) / curr_profit)
                    payback_period = f"{payback:.2f}年"
                else:
                    payback_period = "无法计算" # 收益为0无法回收
            break
    if payback_period == "":
        payback_period = "未回收"

    # 修改表头和值，将“初始投资不含税”放在下一行
    top_headers_row1 = [
        "含税放电", "含税充电", "不含税放电", "不含税充电", "投资度数", "实际度数", "台数", "成本",
        "运行天数", "效率", "备注", "初始投资含税", "EPC16年含税收益",
        "EPC投资回收周期", "运维成本", "所得税", "利息", "EMC折扣"
    ]
    top_values_row1 = [
        discharge_tax, charge_tax, discharge_excl, charge_excl, investment_degree, actual_degree, 台数, cost_per_kwh,
        run_days, efficiency, "第1、2年免运维费用", initial_invest_tax,
        epc_16year_total_profit, payback_period, f"{om_cost_rate*100}%", f"{tax_rate*100}%", f"{interest_rate*100}%", emc_discount
    ]
    top_headers_row2 = ["初始投资不含税"]
    top_values_row2 = [initial_invest_excl]

    # 写入第一行表头和数据
    for col_idx, header in enumerate(top_headers_row1, start=1):
        cell = sheet_profit.cell(row=1, column=col_idx, value=header)
        cell.font = header_font
        cell.alignment = center_align
        cell.border = thin_border
        cell.fill = fill_header_yellow

    for col_idx, val in enumerate(top_values_row1, start=1):
        cell = sheet_profit.cell(row=2, column=col_idx, value=val)
        cell.font = data_font
        cell.alignment = center_align
        cell.border = thin_border
        if isinstance(val, float):
            if col_idx in [1,2,3,4,8,12,13]:
                cell.number_format = '0.00'
            elif col_idx == 9:
                cell.number_format = '0.00'
            elif col_idx == 10:
                cell.number_format = '0.00%'
            elif col_idx == 14:
                cell.number_format = '0.00'
            elif col_idx == 18:
                cell.number_format = '0.00'
        elif isinstance(val, str) and '%' in val:
            pass

    # 写入第二行表头和数据 (初始投资不含税)
    # 计算第二行从哪个列开始
    col_start_for_second_row_header = len(top_headers_row1) - len(top_headers_row2) + 1 # 最后一列前，倒数第二个位置

    cell_header_initial_invest_excl = sheet_profit.cell(row=1, column=col_start_for_second_row_header, value=top_headers_row2[0])
    cell_header_initial_invest_excl.font = header_font
    cell_header_initial_invest_excl.alignment = center_align
    cell_header_initial_invest_excl.border = thin_border
    cell_header_initial_invest_excl.fill = fill_header_yellow

    cell_value_initial_invest_excl = sheet_profit.cell(row=2, column=col_start_for_second_row_header, value=top_values_row2[0])
    cell_value_initial_invest_excl.font = data_font
    cell_value_initial_invest_excl.alignment = center_align
    cell_value_initial_invest_excl.border = thin_border
    if isinstance(top_values_row2[0], float):
        cell_value_initial_invest_excl.number_format = '0.00'

    # 调整 EPC模式业主投资回收期 的起始行
    epc_start_row = 4


    epc_headers = ["年限"] + [str(y) for y in years]
    sheet_profit.merge_cells(start_row=epc_start_row, start_column=1, end_row=epc_start_row, end_column=len(epc_headers))
    title_epc = sheet_profit.cell(row=epc_start_row, column=1, value="EPC模式业主投资回收期")
    title_epc.font = header_font
    title_epc.alignment = center_align
    title_epc.border = thin_border
    title_epc.fill = fill_header_yellow

    header_row = epc_start_row + 1
    for col_idx, header in enumerate(epc_headers, start=1):
        cell = sheet_profit.cell(row=header_row, column=col_idx, value=header)
        cell.font = header_font
        cell.alignment = center_align
        cell.border = thin_border
        cell.fill = fill_header_yellow

    charge_vol_list = []
    discharge_vol_list = []
    income_list = []
    cost_charge_list = []
    om_cost_list = []
    profit_tax_list = []
    roi_list = []
    for i in range(16):
        charge_vol = actual_degree * run_days / 10000 * 2 * battery_cap[i] / (1 - (0.5 - efficiency/2))
        discharge_vol = charge_vol * efficiency
        income = discharge_excl * discharge_vol
        cost_charge = charge_excl * charge_vol
        om_cost = 0 if i < 2 else 台数 * om_cost_rate * 20
        profit_before_tax = income - cost_charge - om_cost
        profit_tax = profit_before_tax * 1.13
        roi = profit_tax / initial_invest_tax # 使用全局的 initial_invest_tax
        charge_vol_list.append(charge_vol)
        discharge_vol_list.append(discharge_vol)
        income_list.append(income)
        cost_charge_list.append(cost_charge)
        om_cost_list.append(om_cost)
        profit_tax_list.append(profit_tax)
        roi_list.append(roi)

    epc_params = [
        ("储能容量(kwh)", [investment_degree]*16, True),
        ("年运行天数(天)", [run_days]*16, True),
        ("系统效率", [efficiency]*16, True),
        ("放电电价(元/kwh)", [discharge_tax]*16, True),
        ("充电电价(元/kwh)", [charge_tax]*16, True),
        ("电池容量", [battery_cap[i] for i in range(16)], False),
        ("充电量(万度)", charge_vol_list, False),
        ("放电量(万度)", discharge_vol_list, False),
        ("发电收入(万元)", income_list, False),
        ("购电成本(万元)", cost_charge_list, False),
        ("运维成本(万元)", om_cost_list, False),
        ("含税收益(万元)", profit_tax_list, False),
        ("ROI", roi_list, False),
        ("净现值(万元)", npv_list, False)
    ]

    data_start_row = header_row + 1
    for row_offset, (param_name, values, is_constant) in enumerate(epc_params):
        excel_row = data_start_row + row_offset
        param_cell = sheet_profit.cell(row=excel_row, column=1, value=param_name)
        param_cell.font = header_font
        param_cell.alignment = center_align
        param_cell.border = thin_border
        param_cell.fill = fill_header_yellow
        if is_constant:
            sheet_profit.merge_cells(start_row=excel_row, start_column=2, end_row=excel_row, end_column=17)
            const_cell = sheet_profit.cell(row=excel_row, column=2, value=values[0])
            const_cell.font = data_font
            const_cell.alignment = center_align
            const_cell.border = thin_border
            if isinstance(values[0], float):
                if param_name in ["系统效率", "电池容量", "ROI"]:
                    const_cell.number_format = '0.00%'
                elif param_name == "净现值(万元)":
                    const_cell.number_format = '0.0'
                else:
                    const_cell.number_format = '0.00'
        else:
            for col_offset, val in enumerate(values, start=2):
                cell = sheet_profit.cell(row=excel_row, column=col_offset, value=val)
                cell.font = data_font
                cell.alignment = center_align
                cell.border = thin_border
                if isinstance(val, float):
                    if param_name in ["电池容量", "ROI"]:
                        cell.number_format = '0.00%'
                    elif param_name == "净现值(万元)":
                        cell.number_format = '0.0'
                    else:
                        cell.number_format = '0.00'

    emc_data_rows = []
    for i in range(16):
        charge_vol = actual_degree * run_days / 10000 * 2 * battery_cap[i] / (1 - (0.5 - efficiency/2))
        discharge_vol = charge_vol * efficiency
        owner_profit = (discharge_vol * discharge_tax - charge_vol * charge_tax) * (1 - emc_discount)
        emc_data_rows.append([
            years[i], investment_degree, run_days, efficiency,
            discharge_tax, charge_tax, emc_discount, 1-emc_discount,
            battery_cap[i], charge_vol, discharge_vol, owner_profit
        ])

    emc_params = [
        ("储能容量(kwh)", [investment_degree]*16, True),
        ("年运行天数(天)", [run_days]*16, True),
        ("系统效率", [efficiency]*16, True),
        ("放电电价(元/kwh)", [discharge_tax]*16, True),
        ("充电电价(元/kwh)", [charge_tax]*16, True),
        ("电价折扣", [emc_discount]*16, True),
        ("储能分成", [1-emc_discount]*16, True),
        ("电池容量", [row[8] for row in emc_data_rows], False),
        ("充电度数(万度)", [row[9] for row in emc_data_rows], False),
        ("放电度数(万度)", [row[10] for row in emc_data_rows], False),
        ("业主含税收益(万元)", [row[11] for row in emc_data_rows], False)
    ]

    cum_10 = sum([row[11] for row in emc_data_rows[:10]])
    cum_12 = sum([row[11] for row in emc_data_rows[:12]])
    cum_15 = sum([row[11] for row in emc_data_rows[:15]])
    cum_16 = sum([row[11] for row in emc_data_rows[:16]])

    emc_start_row = data_start_row + len(epc_params) + 2
    emc_headers = ["年限"] + [str(y) for y in years] + ["10年累计", "12年累计", "15年累计", "16年累计"]

    sheet_profit.merge_cells(start_row=emc_start_row, start_column=1, end_row=emc_start_row, end_column=len(emc_headers))
    title_emc = sheet_profit.cell(row=emc_start_row, column=1, value="EMC模式业主收益情况")
    title_emc.font = header_font
    title_emc.alignment = center_align
    title_emc.border = thin_border
    title_emc.fill = fill_header_yellow

    emc_header_row = emc_start_row + 1
    for col_idx, header in enumerate(emc_headers, start=1):
        cell = sheet_profit.cell(row=emc_header_row, column=col_idx, value=header)
        cell.font = header_font
        cell.alignment = center_align
        cell.border = thin_border
        cell.fill = fill_header_yellow

    emc_data_start_row = emc_header_row + 1
    for row_offset, (param_name, values, is_constant) in enumerate(emc_params):
        excel_row = emc_data_start_row + row_offset
        param_cell = sheet_profit.cell(row=excel_row, column=1, value=param_name)
        param_cell.font = header_font
        param_cell.alignment = center_align
        param_cell.border = thin_border
        param_cell.fill = fill_header_yellow
        if is_constant:
            sheet_profit.merge_cells(start_row=excel_row, start_column=2, end_row=excel_row, end_column=17)
            const_cell = sheet_profit.cell(row=excel_row, column=2, value=values[0])
            const_cell.font = data_font
            const_cell.alignment = center_align
            const_cell.border = thin_border
            if isinstance(values[0], float):
                if param_name in ["系统效率", "电池容量", "电价折扣", "储能分成"]:
                    const_cell.number_format = '0.00%'
                else:
                    const_cell.number_format = '0.00'
            for col in range(18, len(emc_headers)+1):
                sheet_profit.cell(row=excel_row, column=col, value="")
        else:
            for col_offset, val in enumerate(values, start=2):
                cell = sheet_profit.cell(row=excel_row, column=col_offset, value=val)
                cell.font = data_font
                cell.alignment = center_align
                cell.border = thin_border
                if isinstance(val, float):
                    if param_name == "电池容量":
                        cell.number_format = '0.00%'
                    else:
                        cell.number_format = '0.00'
            for col in range(18, len(emc_headers)+1):
                sheet_profit.cell(row=excel_row, column=col, value="")

    cum_row = emc_data_start_row + len(emc_params)
    param_cell = sheet_profit.cell(row=cum_row, column=1, value="累计收益")
    param_cell.font = header_font
    param_cell.alignment = center_align
    param_cell.border = thin_border
    param_cell.fill = fill_header_yellow
    for col in range(2, 18):
        sheet_profit.cell(row=cum_row, column=col, value="")
    cum_vals = [cum_10, cum_12, cum_15, cum_16]
    for col_offset, val in enumerate(cum_vals, start=18):
        cell = sheet_profit.cell(row=cum_row, column=col_offset, value=val)
        cell.font = data_font
        cell.alignment = center_align
        cell.border = thin_border
        cell.number_format = '0.00'

    for col_idx in range(1, len(emc_headers)+1):
        col_letter = get_column_letter(col_idx)
        if col_idx == 1:
            sheet_profit.column_dimensions[col_letter].width = 15
        elif 2 <= col_idx <= 17:
            sheet_profit.column_dimensions[col_letter].width = 8
        else:
            sheet_profit.column_dimensions[col_letter].width = 10

    sheet_profit.row_dimensions[1].height = 25
    sheet_profit.row_dimensions[2].height = 20
    sheet_profit.row_dimensions[epc_start_row].height = 20
    sheet_profit.row_dimensions[header_row].height = 25
    sheet_profit.row_dimensions[emc_start_row].height = 20
    sheet_profit.row_dimensions[emc_header_row].height = 25

    # 4. 设备规格
    sheet_spec = wb.create_sheet("设备规格", 3)
    for c_idx, col_name in enumerate(device_specs.columns, start=1):
        cell = sheet_spec.cell(row=1, column=c_idx, value=col_name)
        cell.font = header_font
        cell.fill = fill_header_yellow
        cell.alignment = center_align
        cell.border = thin_border

    for r_idx, row in enumerate(device_specs.values.tolist(), start=2):
        for c_idx, val in enumerate(row, start=1):
            cell = sheet_spec.cell(row=r_idx, column=c_idx, value=val)
            cell.alignment = center_align
            cell.border = thin_border
            if isinstance(val, (int, float)):
                cell.number_format = '0.00'
            cell.font = data_font

    for col_idx in range(1, len(device_specs.columns)+1):
        sheet_spec.column_dimensions[get_column_letter(col_idx)].width = 12

    # 5. 2充放回收期
    sheet_payback = wb.create_sheet("2充放回收期", 4)
    # 标题行
    sheet_payback.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(payback_df.columns))
    title_cell = sheet_payback.cell(row=1, column=1, value="项目回收期详细测算表")
    title_cell.font = Font(name='微软雅黑', size=14, bold=True)
    title_cell.alignment = center_align
    title_cell.fill = PatternFill(start_color='B4C6E7', end_color='B4C6E7', fill_type='solid') # 蓝色填充

    for r_idx, row in enumerate(payback_df.values):
        for c_idx, val in enumerate(row):
            # 实际数据从第2行开始写入，因为第1行是标题
            cell = sheet_payback.cell(row=r_idx+2, column=c_idx+1, value=val)
            cell.alignment = center_align
            cell.border = thin_border
            if r_idx == 0: # 表头行
                cell.font = header_font
                cell.fill = fill_header_yellow
            else: # 数据行
                cell.font = data_font
                if isinstance(val, (int, float)):
                    cell.number_format = '0.00' if '.' in str(val) else '0'
                elif isinstance(val, str) and (val.endswith('%') or "年" in val or "回收" in val): # 包含%、年、回收等文字的字符串
                    cell.number_format = '@' # 文本格式
                elif isinstance(val, str) and val.replace('.', '', 1).isdigit(): # 看起来是数字的字符串
                    try:
                        float(val)
                        cell.number_format = '0.00'
                    except ValueError:
                        cell.number_format = '@'
                else:
                    cell.number_format = '@' # 其他字符串为文本格式

    # 冻结窗格: 冻结第一行和第一列
    sheet_payback.freeze_panes = 'B3' # B3表示冻结第1行和第1列 (Excel中行和列从1开始)

    for col_idx in range(1, len(payback_df.columns)+1):
        col_letter = get_column_letter(col_idx)
        sheet_payback.column_dimensions[col_letter].width = 12
    sheet_payback.column_dimensions[get_column_letter(1)].width = 18
    sheet_payback.row_dimensions[1].height = 25
    sheet_payback.row_dimensions[2].height = 25 # 表头行高度

    # 6. 回收周期表
    create_sensitivity_analysis_sheet(writer, "回收周期表")

print("✅ 执行完成！已生成【表2数据.xlsx】")
print("📌 最终输出包含以下工作表：")
print("  1. 容量-消纳测算")
print("  2. 容量-储能配置")
print("  3. 2充放收益测算")
print("  4. 设备规格")
print("  5. 2充放回收期")
print("  6. 回收周期表")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_31936\1055494103.py:508: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_raw["时间"] = pd.to_datetime(df_raw["时间"]).dt.strftime("%H:%M:%S")


✅ 执行完成！已生成【表2数据.xlsx】
📌 最终输出包含以下工作表：
  1. 容量-消纳测算
  2. 容量-储能配置
  3. 2充放收益测算
  4. 设备规格
  5. 2充放回收期
  6. 回收周期表
